In [1]:
import optuna
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from xgboost import XGBRegressor

FILE_PATH = Path("daily_demand_features.csv")
df = pd.read_csv(FILE_PATH)

X = df.drop(columns=["Total_Quantity"])
y = df["Total_Quantity"]

# Split data FIRST before mapping to prevent data leakage (shuffle=False preserves time order)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    random_state=42,
    test_size=0.2,
    shuffle=False
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    random_state=42,
    test_size=0.2,
    shuffle=False
)

# Frequency encode StockCode string column ONLY on X_train to prevent leakage
stock_freq = X_train[
    "StockCode"
].value_counts()

X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()

X_train["StockCode"] = X_train[
    "StockCode"
].map(
    stock_freq
).fillna(0)

X_val["StockCode"] = X_val[
    "StockCode"
].map(
    stock_freq
).fillna(0)

X_test["StockCode"] = X_test[
    "StockCode"
].map(
    stock_freq
).fillna(0)

def objective(trail):
    params = {
        "max_depth" : trail.suggest_int(
            "max_depth",
            3,
            6
        ),

        "learning_rate" : trail.suggest_float(
            "learning_rate",
            0.01,
            0.1
        ),

        "n_estimators" : trail.suggest_int(
            "n_estimators",
            200,
            400
        ),
        "random_state" : 42,
        "eval_metric" : "mae",
        "early_stopping_rounds" : 30
    }
    model = XGBRegressor(
        **params
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[
            (
                X_val,
                y_val
            )
        ],
        verbose=False
    )

    val_prediction = model.predict(X_val)

    score = mean_absolute_error(y_val, val_prediction)

    return score

study = optuna.create_study(
    direction="minimize"
)

study.optimize(
    objective,
    n_trials=50
)

best_settings = study.best_params.copy()
best_settings["random_state"] = 42
best_settings["eval_metric"] = "mae"
best_settings["early_stopping_rounds"] = 30

final_model = XGBRegressor(**best_settings)

final_model.fit(
    X_train,
    y_train,
    eval_set=[
        (
            X_val,
            y_val
        )
    ],
    verbose=False
)

pred = final_model.predict(
    X_test
)

print("===== REGRESSION REPORT =====")

print(
    f"MAE  : {mean_absolute_error(y_test, pred):.2f}"
)
print(
    f"RMSE : {np.sqrt(mean_squared_error(y_test, pred)):.2f}"
)
print(
    f"R²   : {r2_score(y_test, pred):.2f}"
)

print("\n===== OPTUNA RESULTS =====")

print(
    f"Best Val MAE : {study.best_value:.2f}"
)

print("\n===== BEST PARAMETERS =====")

for setting_name, setting_value in study.best_params.items():
    print(
        f"{setting_name} : {setting_value}"
    )

C:\Users\HP\PyCharmMiscProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-09-14 18:48:07,434] A new study created in memory with name: no-name-bdaf82e3-f04a-47e1-9db5-3f1a1ed932e9
[I 2026-09-14 18:48:08,753] Trial 0 finished with value: 18.466690063476562 and parameters: {'max_depth': 4, 'learning_rate': 0.09034646766616146, 'n_estimators': 277}. Best is trial 0 with value: 18.466690063476562.
[I 2026-09-14 18:48:09,315] Trial 1 finished with value: 19.448740005493164 and parameters: {'max_depth': 3, 'learning_rate': 0.08303337765397859, 'n_estimators': 384}. Best is trial 0 with value: 18.466690063476562.
[I 2026-09-14 18:48:09,906] Trial 2 finished with value: 19.890844345092773 and parameters: {'max_depth': 4, 'learning_rate': 0.054091915124699894, 'n_estimators': 233}. Best is trial 0 with v

===== REGRESSION REPORT =====
MAE  : 17.8606
RMSE : 55.3650
R²   : 0.1588

===== OPTUNA RESULTS =====
Best Val MAE : 18.3175

===== BEST PARAMETERS =====
max_depth : 4
learning_rate : 0.09636387778000224
n_estimators : 285
